# 옵티마이저와 드롭아웃 

- 옵티마이저는 기계 학습에서 모델의 학습 과정을 조절하는 알고리즘
- 경사하강법의 원리를 최적화 하여 손실 함수의 최소 값을 찾기 위해 모델의 매개변수를 조정하는 역할 
- 옵티마이저는 모델의 성능을 최적화하기 위해 손실 함수의 기울기(미분값)을 사용하여 매개변수를 어떻게 업데이트 할 지 결정 
- 학습률 조정, 지역 최소값 문제 해결, 학습과정 가속화 등을 포함하여, 경사 하강법을 더 효율적으로 적용하는 방법을 제공 

## 장점

- 학습 과정을 가속화, 학습 시간을 단축 
- 지역 최소값에서 벗어나 전역 최소값에 더 가까워질 수 있도록 도와준다. 
- 학습률을 자동으로 조절하여 최적의 학습률을 찾을 수 있다. 

## 단점 

- 옵티마이저 선택이 모델의 성능에 큰 영향을 미칠 수 있으며, 때로는 실험을 통해 최적의 옵티마이저를 찾아야 할 수도 있다. 
- 과적합(Overfitting)을 방지하기 위한 추가적인 조치가 필요 할 수 있다. 

## 주의사항

- 과적합을 방지하기 위해 조기 종료(Early Stopping), 드롭 아웃(Dropout) 등의 기법과 함께 사용하는 것이 좋다.  

## SGD

- 아이디어: 현재 기울기 방향으로 일정한 크기(`lr`)만큼 이동
- 장점: 구조가 단순하고 메모리 사용량이 작음
- 단점: 지그재그로 진동하거나, 학습률에 매우 민감함
- 언제 쓰나: 매우 큰 모델에서 메모리 절약이 중요하거나, 모멘텀과 함께 기본 베이스라인을 만들 때

## Adagrad (Adaptive Gradient Algorithm)

- 아이디어: 파라미터별로 과거 기울기 제곱합을 누적해, 자주 업데이트된 파라미터의 학습률을 자동으로 줄임
- 장점: 희소(sparse)한 특성에서 유리함 (예: NLP의 희소 입력)
- 단점: 학습이 진행될수록 학습률이 너무 작아져 학습이 일찍 멈출 수 있음
- 언제 쓰나: 희소 피처가 많은 문제에서 시도해볼 가치가 있음

## RMSprop (Root Mean Square Propagation)

- 아이디어: Adagrad의 누적 방식 대신 최근 기울기 제곱의 지수이동평균을 사용해 학습률 감소를 완화
- 장점: 비정상(non-stationary) 손실 환경에서 안정적, RNN류에서 자주 사용됨
- 단점: 하이퍼파라미터(`lr`, `alpha`)에 따라 성능 차이가 큼
- 언제 쓰나: 손실 곡선이 흔들리거나 Adagrad가 너무 빨리 멈출 때

## 요즘 실무에서 많이 쓰는 옵티마이저: AdamW

- Adam + Weight Decay를 올바르게 분리한 방식
- 장점: Adam의 빠른 수렴 + 일반화 성능 개선(과적합 완화)
- 현재 트랜스포머/비전 모델 등에서 기본 선택지로 매우 널리 사용

### PyTorch 예제

```python
import torch
import torch.nn as nn
import torch.optim as optim

model = nn.Linear(10, 1)

# 1) SGD
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 2) Adagrad
optimizer_adagrad = optim.Adagrad(model.parameters(), lr=0.01)

# 3) RMSprop
optimizer_rmsprop = optim.RMSprop(model.parameters(), lr=0.001, alpha=0.99)

# 4) AdamW (최근 실무 기본 선택지)
optimizer_adamw = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
```

### 간단 선택 가이드

- 빠르고 무난한 시작: AdamW
- 메모리 절약/고전 베이스라인: SGD(+momentum)
- 희소 입력: Adagrad
- 손실 진동이 큰 경우: RMSprop

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 예제 모델
model = nn.Linear(10, 1)

# 요청한 3가지 + 최신 실무에서 자주 쓰는 AdamW
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
optimizer_adagrad = optim.Adagrad(model.parameters(), lr=0.01)
optimizer_rmsprop = optim.RMSprop(model.parameters(), lr=0.001, alpha=0.99)
optimizer_adamw = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

print("SGD:", optimizer_sgd)
print("Adagrad:", optimizer_adagrad)
print("RMSprop:", optimizer_rmsprop)
print("AdamW:", optimizer_adamw)

SGD: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)
Adagrad: Adagrad (
Parameter Group 0
    differentiable: False
    eps: 1e-10
    foreach: None
    fused: None
    initial_accumulator_value: 0
    lr: 0.01
    lr_decay: 0
    maximize: False
    weight_decay: 0
)
RMSprop: RMSprop (
Parameter Group 0
    alpha: 0.99
    capturable: False
    centered: False
    differentiable: False
    eps: 1e-08
    foreach: None
    lr: 0.001
    maximize: False
    momentum: 0
    weight_decay: 0
)
AdamW: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.01
)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 재현성을 위해 시드 고정
torch.manual_seed(42)

# 간단한 선형 데이터: y = 2x
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[2.0], [4.0], [6.0], [8.0]])


def train_with_optimizer(
    optimizer_cls,
    optimizer_kwargs,
    x_tensor,
    y_tensor,
    epochs=10,
    lr=0.01,
):
    """optimizer 클래스 참조를 받아 동일한 학습 절차를 재사용하는 함수"""
    model = nn.Linear(1, 1)
    optimizer = optimizer_cls(model.parameters(), lr=lr, **optimizer_kwargs)

    print(f"\n=== {optimizer_cls.__name__} (lr={lr}) ===")

    for epoch in range(epochs + 1):  # 0~10 epoch
        pred = model(x_tensor)
        loss = F.mse_loss(pred, y_tensor)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # 0, 5, 10 epoch만 출력해서 변화 확인
        if epoch in (0, 5, 10):
            w = model.weight.item()
            b = model.bias.item()
            print(
                f"Epoch {epoch:>2} | loss={loss.item():.6f} | "
                f"w={w:.4f}, b={b:.4f}"
            )

    return model, optimizer


# optimizer 클래스 레퍼런스를 넘겨 재사용
optimizer_configs = [
    (optim.SGD, {"momentum": 0.9}, 0.01),
    (optim.Adagrad, {}, 0.05),
    (optim.RMSprop, {"alpha": 0.99}, 0.01),
    (optim.AdamW, {"weight_decay": 1e-2}, 0.01),
]

trained_objects = {}
for opt_cls, opt_kwargs, lr in optimizer_configs:
    trained_model, trained_optimizer = train_with_optimizer(
        optimizer_cls=opt_cls,
        optimizer_kwargs=opt_kwargs,
        x_tensor=x,
        y_tensor=y,
        epochs=10,
        lr=lr,
    )
    trained_objects[opt_cls.__name__] = {
        "model": trained_model,
        "optimizer": trained_optimizer,
    }

print("\n학습 완료: SGD / Adagrad / RMSprop / AdamW")


=== SGD (lr=0.01) ===
Epoch  0 | loss=7.009437 | w=0.9084, b=0.8752
Epoch  5 | loss=1.530008 | w=2.1755, b=1.2438
Epoch 10 | loss=0.582197 | w=1.6787, b=0.9659

=== Adagrad (lr=0.05) ===
Epoch  0 | loss=28.021507 | w=-0.1843, b=0.9686
Epoch  5 | loss=22.309702 | w=-0.0567, b=1.0958
Epoch 10 | loss=19.553106 | w=0.0219, b=1.1739

=== RMSprop (lr=0.01) ===
Epoch  0 | loss=34.734901 | w=-0.1191, b=0.3018
Epoch  5 | loss=22.649422 | w=0.1303, b=0.5502
Epoch 10 | loss=17.435200 | w=0.2808, b=0.6992

=== AdamW (lr=0.01) ===
Epoch  0 | loss=39.425827 | w=-0.4768, b=0.5972
Epoch  5 | loss=37.174973 | w=-0.4266, b=0.6469
Epoch 10 | loss=34.999195 | w=-0.3766, b=0.6963

학습 완료: SGD / Adagrad / RMSprop / AdamW
